# Neural Hydrology — A/B/C Run on Colab

Single-click run of the locked A/B/C protocol on Component 0.

## Configuration

Two variables to adjust if needed (Cell 2):
- `DRIVE_CAMELS_PATH` — where your `camels_us/` folder lives on Drive
- `MODE` — `'demo'` (1 seed × 3 conditions) or `'full'` (5 seeds × 3 conditions)

## How to use

1. Runtime → Change runtime type → **L4 GPU** (or T4 / A100). Save.
2. Runtime → Run all.
3. Wait for Cell 12 to print the `summary.json`.

Notebook is **idempotent** — every condition × seed checks for completed outputs before retraining.

## What's fixed since the previous version

- Cell 8 deletes its smoke-test folder immediately after the smoke test passes, so it never gets matched by later skip-if-done globs.
- Cell 9 calls `nh_run.py evaluate` after `train` so Condition A's `test_metrics.csv` exists for Cell 12.
- Cells 10 and 11 explicitly filter out any `_SMOKE_` folder from the skip-if-done check (defense in depth).
- Cell 12 looks for A under both `test/model_epoch030/` and the alternate path NH might write.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Configuration

In [ ]:
import os

# === USER CONFIG ===
GITHUB_URL = 'https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH = ''  # leave empty for auto-detection
MODE = 'demo'           # 'demo' (1 seed) or 'full' (5 seeds)
# ====================

AUTO_DETECT_CANDIDATES = [
    '/content/drive/MyDrive/datasets/camels_us',
    '/content/drive/MyDrive/neural_hydro/datasets/camels_us',
    '/content/drive/MyDrive/neural_hydrology/datasets/camels_us',
    '/content/drive/MyDrive/camels_us',
    '/content/drive/MyDrive/data/camels_us',
]
if not DRIVE_CAMELS_PATH:
    for cand in AUTO_DETECT_CANDIDATES:
        if os.path.isdir(cand):
            DRIVE_CAMELS_PATH = cand
            print(f'Auto-detected camels_us at: {cand}')
            break
    else:
        raise RuntimeError(
            'Could not find camels_us. Set DRIVE_CAMELS_PATH explicitly.')
else:
    assert os.path.isdir(DRIVE_CAMELS_PATH)

topo_file = os.path.join(DRIVE_CAMELS_PATH, 'camels_attributes_v2.0', 'camels_topo.txt')
assert os.path.isfile(topo_file), f'Expected {topo_file} — does the folder have CAMELS contents?'
print(f'Verified camels_topo.txt present.')

DRIVE_RUNS = '/content/drive/MyDrive/neural_hydrology_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print(f'Runs dir: {DRIVE_RUNS}')

SEEDS = [42] if MODE == 'demo' else [11, 13, 17, 19, 23]
print(f'\nMODE = {MODE}; SEEDS = {SEEDS}')

## Cell 3 — Clone the repo from GitHub

In [ ]:
REPO_DIR = '/content/nh'
import shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 3

## Cell 4 — Install dependencies (pin numpy<2 so torch C-bindings work)

In [ ]:
%cd {REPO_DIR}
# Install in two passes to avoid the numpy/pandas ABI mismatch we hit before:
# pin numpy<2 first, install everything else, then re-pin numpy and pandas
# to a compatible pair.
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2

# Verify
import importlib, sys
for mod in list(sys.modules):
    if mod.startswith('numpy') or mod.startswith('pandas'):
        del sys.modules[mod]
import numpy as np, pandas as pd, torch
x = torch.from_numpy(np.array([1.0]))
_ = pd.DataFrame({'a': [1, 2]})
print(f'numpy {np.__version__}  pandas {pd.__version__}  torch {torch.__version__}  CUDA: {torch.cuda.is_available()}')

## Cell 5 — Symlink data and runs from Drive

In [ ]:
%cd {REPO_DIR}
REPO_DATA = os.path.join(REPO_DIR, 'datasets', 'camels_us')
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
if os.path.islink(REPO_DATA) or os.path.isdir(REPO_DATA):
    !rm -rf {REPO_DATA}
os.symlink(DRIVE_CAMELS_PATH, REPO_DATA)

REPO_RUNS = os.path.join(REPO_DIR, 'runs')
if os.path.islink(REPO_RUNS) or os.path.isdir(REPO_RUNS):
    !rm -rf {REPO_RUNS}
os.symlink(DRIVE_RUNS, REPO_RUNS)

print(f'datasets/camels_us -> {DRIVE_CAMELS_PATH}')
print(f'runs/ -> {DRIVE_RUNS}')
!ls datasets/camels_us | head -3

## Cell 6 — GPU check

In [ ]:
!nvidia-smi -L
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> select a GPU.')
print(f'GPU: {torch.cuda.get_device_name(0)}, '
       f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 7 — Generate per-seed YAML configs for Condition A

In [ ]:
%cd {REPO_DIR}
CONFIG_DIR = os.path.join(REPO_DIR, 'experiments', 'configs', '_seed_configs')
os.makedirs(CONFIG_DIR, exist_ok=True)
BASE_CONFIG = open('experiments/configs/lstm_component0_baseline.yaml').read()
for seed in SEEDS:
    cfg = BASE_CONFIG.replace('experiment_name: lstm_component0_baseline',
                                f'experiment_name: A_baseline_seed{seed}')
    cfg = cfg.replace('device: cpu', 'device: cuda:0')
    if 'seed:' not in cfg:
        cfg += f'\nseed: {seed}\n'
    out = os.path.join(CONFIG_DIR, f'A_seed{seed}.yaml')
    with open(out, 'w') as f:
        f.write(cfg)
print(f'Wrote {len(SEEDS)} seed configs')

## Cell 8 — Smoke test (catches setup errors), then auto-cleanup

Trains the 23-basin pilot baseline (one-time prerequisite for B/C cfg) and runs a 2-epoch graph-LSTM smoke test. **The smoke-test folder is deleted immediately after** so it doesn't pollute the skip-if-done logic in later cells.

In [ ]:
%cd {REPO_DIR}
import glob, shutil, time

# (1) 23-basin pilot baseline — one-time; B/C use it for cfg + scaler
if not glob.glob(f'{REPO_DIR}/runs/05_lstm_23basin_strong_baseline'):
    print('Training 23-basin pilot baseline as a one-time prerequisite...')
    t0 = time.time()
    !python neuralhydrology/nh_run.py train --config-file experiments/configs/lstm_study_network_strong.yaml 2>&1 | tail -2
    pilot = sorted(glob.glob(f'{REPO_DIR}/runs/lstm_study_network_strong_*'))[-1]
    !mv {pilot} {REPO_DIR}/runs/05_lstm_23basin_strong_baseline
    print(f'    23-basin baseline done in {(time.time() - t0)/60:.1f} min')
else:
    print('23-basin pilot baseline already exists — skipping.')

# (2) Smoke test of graph-LSTM training pipeline
print('\nSmoke-testing graph-LSTM training (2 epochs on 23 basins)...')
!python experiments/training/train_graph_component0.py \
    --variant warm \
    --seed 42 \
    --smoke-test \
    --no-warm-start \
    --basin-file experiments/basin_lists/study_network_basins.txt \
    --edge-file topology_analysis/phase1_network_discovery/outputs/study_network_edges.csv \
    --baseline-run runs/05_lstm_23basin_strong_baseline 2>&1 | tail -3

# (3) IMMEDIATE cleanup: remove the SMOKE folder so its test_metrics.csv
# can never be confused with a real Condition C run.
smoke_dirs = glob.glob(f'{REPO_DIR}/runs/graph_c0_warm_seed42_SMOKE_*')
for d in smoke_dirs:
    shutil.rmtree(d)
    print(f'  cleaned up: {d}')
print(f'\nSmoke-test cleanup done. {len(smoke_dirs)} smoke folder(s) removed.')

## Cell 9 — Condition A: NH cudalstm baseline (no graph), train + evaluate × seeds

In [ ]:
%cd {REPO_DIR}
import time, glob

for seed in SEEDS:
    # Skip if both train AND test eval are done
    if glob.glob(f'{REPO_DIR}/runs/A_baseline_seed{seed}_*/test/model_epoch030/test_metrics.csv'):
        print(f'[skip] A seed={seed} already complete (train + evaluate)')
        continue

    cfg = f'{CONFIG_DIR}/A_seed{seed}.yaml'
    print(f'\n=== Condition A — seed={seed} (TRAIN) ===')
    t0 = time.time()

    # Train (only if not already trained)
    if not glob.glob(f'{REPO_DIR}/runs/A_baseline_seed{seed}_*/model_epoch030.pt'):
        !python neuralhydrology/nh_run.py train --config-file {cfg} 2>&1 | tail -2
    else:
        print('    (already trained; running evaluate only)')

    # Then evaluate to produce test_metrics.csv (required by Cell 12)
    A_dir = sorted(glob.glob(f'{REPO_DIR}/runs/A_baseline_seed{seed}_*'))[-1]
    print(f'=== Condition A — seed={seed} (EVALUATE on {os.path.basename(A_dir)}) ===')
    !python neuralhydrology/nh_run.py evaluate --run-dir {A_dir} --epoch 30 2>&1 | tail -3
    print(f'    {(time.time() - t0)/60:.1f} min total')

## Cell 10 — Condition B: graph-LSTM with topology features, no message passing

In [ ]:
%cd {REPO_DIR}
BASELINE_FOR_BC = sorted(glob.glob(f'{REPO_DIR}/runs/A_baseline_seed*/'))[0]
print(f'Using {BASELINE_FOR_BC} for B/C cfg + scaler')

for seed in SEEDS:
    # Skip-if-done check filters out any SMOKE folder explicitly (defense in depth)
    existing = [d for d in glob.glob(f'{REPO_DIR}/runs/graph_c0_topology_features_seed{seed}_*')
                 if 'SMOKE' not in d
                 and os.path.isfile(os.path.join(d, 'test_metrics.csv'))]
    if existing:
        print(f'[skip] B seed={seed} already complete: {os.path.basename(existing[0])}')
        continue
    print(f'\n=== Condition B — seed={seed} ===')
    t0 = time.time()
    !python experiments/training/train_graph_component0.py \
        --variant topology_features \
        --seed {seed} --no-warm-start --epochs 30 \
        --baseline-run {BASELINE_FOR_BC} 2>&1 | tail -3
    print(f'    {(time.time() - t0)/60:.1f} min')

## Cell 11 — Condition C: full graph-LSTM (edges + message passing)

Skip-if-done explicitly excludes any `_SMOKE_` folder so smoke-test artifacts can't fool the check.

In [ ]:
%cd {REPO_DIR}
for seed in SEEDS:
    existing = [d for d in glob.glob(f'{REPO_DIR}/runs/graph_c0_warm_seed{seed}_*')
                 if 'SMOKE' not in d
                 and os.path.isfile(os.path.join(d, 'test_metrics.csv'))]
    if existing:
        print(f'[skip] C seed={seed} already complete: {os.path.basename(existing[0])}')
        continue
    print(f'\n=== Condition C — seed={seed} ===')
    t0 = time.time()
    !python experiments/training/train_graph_component0.py \
        --variant warm \
        --seed {seed} --no-warm-start --epochs 30 \
        --baseline-run {BASELINE_FOR_BC} 2>&1 | tail -3
    print(f'    {(time.time() - t0)/60:.1f} min')

## Cell 12 — Aggregate results

In [ ]:
%cd {REPO_DIR}
import json, re, glob, os
import pandas as pd, numpy as np
from pathlib import Path

def load_metrics(patterns, exclude='SMOKE'):
    out = {}
    for pat in patterns:
        for p in sorted(glob.glob(pat)):
            if exclude and exclude in p:
                continue
            m = re.search(r'seed(\d+)', p)
            if not m:
                continue
            seed = int(m.group(1))
            if seed in out:
                continue  # first match wins
            df = pd.read_csv(p, dtype={'basin': str})
            out[seed] = df.set_index('basin')['NSE'].to_dict()
    return out

results = {
    'A_baseline': load_metrics([
        f'{REPO_DIR}/runs/A_baseline_seed*/test/model_epoch030/test_metrics.csv',
    ]),
    'B_topology_features': load_metrics([
        f'{REPO_DIR}/runs/graph_c0_topology_features_seed*/test_metrics.csv',
    ]),
    'C_graph_messages': load_metrics([
        f'{REPO_DIR}/runs/graph_c0_warm_seed*/test_metrics.csv',
    ]),
}

# Tell the user what was found per condition (helps catch the previous-class bugs)
for label, r in results.items():
    if r:
        print(f'{label}: {len(r)} seed(s) found: {sorted(r.keys())}')
    else:
        print(f'{label}: NO RESULTS FOUND')

summary = {}
for label, r in results.items():
    if not r:
        continue
    medians = sorted(np.median(list(d.values())) for d in r.values())
    summary[label] = {
        'n_seeds': len(medians),
        'seeds': sorted(r.keys()),
        'median_NSE_per_seed': [float(x) for x in medians],
        'cross_seed_median': float(np.median(medians)),
        'cross_seed_std': float(np.std(medians)) if len(medians) > 1 else 0.0,
    }
print('\n' + json.dumps(summary, indent=2))

# Save
OUT_DIR = Path(REPO_DIR) / 'experiments' / 'analysis_outputs' / 'abc_publication'
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
rows = [{'condition': label, 'seed': s, 'basin': b, 'NSE': v}
        for label, r in results.items() for s, d in r.items() for b, v in d.items()]
if rows:
    pd.DataFrame(rows).to_csv(OUT_DIR / 'per_basin_per_seed.csv', index=False)
    print(f'\nWrote {len(rows)} rows to per_basin_per_seed.csv')

# Headline
if {'A_baseline', 'B_topology_features', 'C_graph_messages'}.issubset(summary):
    a = summary['A_baseline']['cross_seed_median']
    b = summary['B_topology_features']['cross_seed_median']
    c = summary['C_graph_messages']['cross_seed_median']
    print(f'\n*** A baseline (no graph)            : {a:+.3f}')
    print(f'*** B topology-features (no msg pass): {b:+.3f}    B - A = {b - a:+.3f}')
    print(f'*** C full graph-LSTM (msg passing)  : {c:+.3f}    C - A = {c - a:+.3f}    C - B = {c - b:+.3f}')
else:
    missing = {'A_baseline', 'B_topology_features', 'C_graph_messages'} - set(summary.keys())
    print(f'\nMissing conditions: {missing}. Re-run those cells.')

## Done

Result files are at:
- `/content/drive/MyDrive/neural_hydrology_runs/experiments/analysis_outputs/abc_publication/summary.json`
- `/content/drive/MyDrive/neural_hydrology_runs/experiments/analysis_outputs/abc_publication/per_basin_per_seed.csv`

Pull these to your local repo (drag from Drive desktop sync, or copy via Drive web UI). Then in chat type **`crs interpret abc results`** and the chief-research-scientist will read them.